# House Price Prediction Training Notebook

This is the production-ready training notebook for web application integration.

The original Lab 3 notebook compared Linear Regression, Ridge, and Lasso. For deployment, this notebook uses only one final algorithm: **Ridge Regression**. Ridge was selected because the Lab 3 comparison showed strong overall performance with regularization, making it a stable choice for production use.

# Stage 1
Import Libraries

In [2]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TARGET_COLUMN = "median_house_value"
CATEGORICAL_COLUMNS = ["ocean_proximity"]

current_dir = Path.cwd()
if (current_dir / "dataset.csv").exists():
    TRAINING_DIR = current_dir
else:
    TRAINING_DIR = current_dir.parent

DATASET_PATH = TRAINING_DIR / "dataset.csv"
MODEL_DIR = TRAINING_DIR / "saved_models"
MODEL_PATH = MODEL_DIR / "house_price_model.pkl"
SCALER_PATH = MODEL_DIR / "scaler.pkl"

# Stage 2
Load Dataset

In [3]:
data = pd.read_csv(DATASET_PATH)

print(f"Dataset shape: {data.shape}")
data.head()

Dataset shape: (20640, 10)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


# Stage 3
Data Exploration

In [4]:
print("Column information:")
print(data.info())

print("Missing values:")
print(data.isnull().sum())

print("Target summary:")
print(data[TARGET_COLUMN].describe())

print("Ocean proximity values:")
print(data["ocean_proximity"].value_counts())

Column information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB
None
Missing values:
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
me

# Stage 4
Data Cleaning

In [5]:
cleaned_data = data.copy()

numeric_columns = cleaned_data.select_dtypes(include=["number"]).columns.tolist()
numeric_features = [column for column in numeric_columns if column != TARGET_COLUMN]
numeric_medians = cleaned_data[numeric_features].median()

cleaned_data[numeric_features] = cleaned_data[numeric_features].fillna(numeric_medians)
cleaned_data[CATEGORICAL_COLUMNS] = cleaned_data[CATEGORICAL_COLUMNS].fillna("UNKNOWN")

print("Missing values after cleaning:")
print(cleaned_data.isnull().sum())

Missing values after cleaning:
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64


# Stage 5
Feature Selection

In [6]:
features = cleaned_data.drop(columns=[TARGET_COLUMN])
target = cleaned_data[TARGET_COLUMN]

encoded_features = pd.get_dummies(features, columns=CATEGORICAL_COLUMNS)
feature_columns = encoded_features.columns.tolist()

print(f"Total input features after encoding: {len(feature_columns)}")
encoded_features.head()

Total input features after encoding: 13


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,False,False,False,True,False
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,False,False,False,True,False
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,False,False,False,True,False
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,False,False,False,True,False
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,False,False,False,True,False


# Stage 6
Train/Test Split

In [7]:
x_train, x_test, y_train, y_test = train_test_split(
    encoded_features,
    target,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print(f"Training rows: {x_train.shape[0]}")
print(f"Testing rows: {x_test.shape[0]}")

Training rows: 16512
Testing rows: 4128


# Stage 7
Feature Scaling

In [8]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

print("Feature scaling completed.")

Feature scaling completed.


# Stage 8
Model Training

In [9]:
model = Ridge(alpha=0.001)
model.fit(x_train_scaled, y_train)

print("Ridge Regression model trained successfully.")

Ridge Regression model trained successfully.


# Stage 9
Model Evaluation

In [10]:
test_predictions = model.predict(x_test_scaled)

mse = mean_squared_error(y_test, test_predictions)
rmse = mse ** 0.5
mae = mean_absolute_error(y_test, test_predictions)
r2 = r2_score(y_test, test_predictions)

results = pd.DataFrame([
    {
        "Model": "Ridge Regression",
        "R2 Score": round(r2, 4),
        "RMSE": round(rmse, 2),
        "MAE": round(mae, 2),
    }
])

results

,Model,R2 Score,RMSE,MAE
0,Ridge Regression,0.6254,70060.52,50670.74


# Stage 10
Select Best Model

This section prepares the trained model for deployment inside a web application.

The original Lab 3 compared Linear Regression, Ridge, and Lasso. Ridge Regression is selected for deployment because it provided strong overall performance and uses regularization, which helps reduce unstable coefficient behavior when features are related to each other.

For deployment, only one algorithm should be saved. The backend should load one final model instead of running model comparison during user prediction requests.

In [11]:
# This section prepares the trained model for deployment inside a web application.
selected_model_name = "Ridge Regression"
selected_alpha = 0.001

print(f"Selected deployment model: {selected_model_name}")
print(f"Selected alpha value: {selected_alpha}")

Selected deployment model: Ridge Regression
Selected alpha value: 0.001


# Stage 11
Train Final Model

This section prepares the trained model for deployment inside a web application.

After evaluation is complete, the final model is trained again using the full cleaned dataset. This allows the saved model to learn from all available training rows before it is used by the backend.

In [12]:
# This section prepares the trained model for deployment inside a web application.
# A new scaler is fitted on the full training dataset because this scaler will be saved
# and reused later by the backend during prediction.
final_scaler = StandardScaler()
final_features_scaled = final_scaler.fit_transform(encoded_features)

final_model = Ridge(alpha=selected_alpha)
final_model.fit(final_features_scaled, target)

print("Final Ridge Regression model trained on the full cleaned dataset.")

Final Ridge Regression model trained on the full cleaned dataset.


# Stage 12
Save Model

This section prepares the trained model for deployment inside a web application.

`house_price_model.pkl` is saved because the backend needs a trained model file to make predictions without retraining. The file stores the learned Ridge Regression coefficients and useful metadata such as feature column names.

In [13]:
# This section prepares the trained model for deployment inside a web application.
# model.pkl is saved so the backend can load the trained model and predict house prices.
MODEL_DIR.mkdir(parents=True, exist_ok=True)

final_model.feature_columns_ = feature_columns
final_model.numeric_medians_ = numeric_medians.to_dict()
final_model.categorical_columns_ = CATEGORICAL_COLUMNS
final_model.target_column_ = TARGET_COLUMN
final_model.metrics_ = {
    "r2_score": r2,
    "rmse": rmse,
    "mae": mae,
}

joblib.dump(final_model, MODEL_PATH)
print(f"Model saved to: {MODEL_PATH}")

Model saved to: c:\Users\us\Desktop\houseai\training\saved_models\house_price_model.pkl


# Stage 13
Save Scaler

This section prepares the trained model for deployment inside a web application.

`scaler.pkl` is saved because the model was trained on scaled feature values. During prediction, the backend must scale user input using the same scaler before sending the input to the model.

In [14]:
# This section prepares the trained model for deployment inside a web application.
# scaler.pkl is saved so new backend input is transformed in the same way as training data.
joblib.dump(final_scaler, SCALER_PATH)
print(f"Scaler saved to: {SCALER_PATH}")

Scaler saved to: c:\Users\us\Desktop\houseai\training\saved_models\scaler.pkl


# Stage 14
Prepare Model for Web Application

This section prepares the trained model for deployment inside a web application.

The dataset is no longer needed after training because the model has already learned the relationship between input features and house prices. A backend API only needs the saved model, the saved scaler, and the new user input.

Web application prediction flow:

1. The frontend sends house details through a form.
2. The backend receives the request.
3. The backend formats the request into the same feature columns used during training.
4. The backend loads `scaler.pkl` and scales the input.
5. The backend loads `house_price_model.pkl` and predicts the house price.
6. The backend returns the predicted price to the frontend.

In [15]:
# This section prepares the trained model for deployment inside a web application.
# Example backend inference steps:
# 1. Load house_price_model.pkl with joblib.load().
# 2. Load scaler.pkl with joblib.load().
# 3. Convert frontend form data into a pandas DataFrame.
# 4. Apply the same one-hot encoding and column order used during training.
# 5. Scale the input with scaler.pkl.
# 6. Pass the scaled input into model.predict().

print("Training artifacts are ready for backend integration.")
print("The backend can now use house_price_model.pkl and scaler.pkl for inference.")

Training artifacts are ready for backend integration.
The backend can now use house_price_model.pkl and scaler.pkl for inference.
